# 🎬 VideoKit - Professional Video Processing Toolkit

基于Google Colab的专业视频处理工具，支持视频风格迁移、内容融合、画质增强等功能。

## 🚀 快速开始

### 准备工作
1. 点击菜单栏 **Runtime > Change runtime type**
2. 选择 **Hardware accelerator** 为 **GPU**
3. 点击 **Connect** 连接到运行时

### 执行顺序
按顺序执行以下所有代码单元（点击左侧▶️或按 Shift+Enter）

In [ ]:
#@title 🔧 检查GPU配置
!nvidia-smi

import torch
print(f"\nPyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ 未检测到GPU，请确保已选择GPU运行时")

In [ ]:
#@title 📦 安装系统依赖
!apt-get update -y && apt-get install -y ffmpeg git curl wget python3-venv
!pip install --upgrade pip

In [ ]:
#@title 📥 克隆项目仓库
%cd /content
!rm -rf video
!git clone https://github.com/tjwcold/video.git
%cd video
!ls -la

In [ ]:
#@title 📋 安装Python依赖
# 安装基础依赖
!pip install -r requirements.txt

# 卸载可能冲突的onnxruntime版本
!pip uninstall -y onnxruntime onnxruntime-gpu

# 安装与Colab CUDA 12兼容的onnxruntime-gpu版本
!pip install onnxruntime-gpu==1.16.3

In [ ]:
#@title ✅ 验证ONNX Runtime安装
import onnxruntime
print(f"ONNX Runtime版本: {onnxruntime.__version__}")
print(f"可用执行提供程序: {onnxruntime.get_available_providers()}")

if 'CUDAExecutionProvider' in onnxruntime.get_available_providers():
    print("\n🎉 CUDA支持已启用!")
else:
    print("\n⚠️ CUDA支持未启用，将使用CPU模式")
    print("尝试重新安装onnxruntime-gpu...")
    !pip install onnxruntime-gpu==1.16.3 --force-reinstall

In [ ]:
#@title 🔧 运行Colab补丁（修复兼容性问题）
import os
import re

# 补丁1: 修复 translator.py 默认模块名
with open('videokit/translator.py', 'r') as f:
    content = f.read()
content = content.replace("module_name : str = 'VideoKit'", "module_name : str = 'videokit'")
# 添加错误处理
if 'if module_name not in LOCALE_POOL_SET:\n\t\t__autoload__(module_name)\n\n\tcurrent =' not in content:
    content = re.sub(r'if module_name not in LOCALE_POOL_SET:\n\t\t__autoload__\(module_name\)\n\n\tcurrent =', \
                    'if module_name not in LOCALE_POOL_SET:\n\t\t__autoload__(module_name)\n\n\tif module_name not in LOCALE_POOL_SET:\n\t\treturn None\n\n\tcurrent =', content)
with open('videokit/translator.py', 'w') as f:
    f.write(content)
print('✓ 修复 translator.py')

# 补丁2: 修复 __init__.py 添加初始化
with open('videokit/__init__.py', 'w') as f:
    f.write('''from videokit.locales import LOCALES
from videokit.translator import load

load(LOCALES, 'videokit')
''')
print('✓ 修复 __init__.py')

# 补丁3: 修复 clear_static_faces 导入
for fname in ['videokit/benchmarker.py', 'videokit/uis/components/target.py', 'videokit/uis/components/trim_frame.py']:
    if os.path.exists(fname):
        with open(fname, 'r') as f:
            content = f.read()
        content = content.replace('clear_static_faces', 'clear_static_regions')
        with open(fname, 'w') as f:
            f.write(content)
        print(f'✓ 修复 {fname}')

# 补丁4: 修复 quality_enhancer 函数名
if os.path.exists('videokit/processors/modules/quality_enhancer/core.py'):
    with open('videokit/processors/modules/quality_enhancer/core.py', 'r') as f:
        content = f.read()
    content = content.replace('scale_face', 'scale_region')
    content = content.replace('select_faces', 'select_regions')
    content = content.replace('warp_face_by_face_landmark_5', 'warp_region_by_region_landmark_5')
    with open('videokit/processors/modules/quality_enhancer/core.py', 'w') as f:
        f.write(content)
    print('✓ 修复 quality_enhancer/core.py')

# 补丁5: 修复 UI 组件类型名
for fname, old, new in [
    ('videokit/uis/components/region_detector.py', 'FaceDetectorModel', 'RegionDetectorModel'),
    ('videokit/uis/components/landmark_detector.py', 'FaceLandmarkerModel', 'RegionLandmarkerModel'),
]:
    if os.path.exists(fname):
        with open(fname, 'r') as f:
            content = f.read()
        content = content.replace(old, new)
        with open(fname, 'w') as f:
            f.write(content)
        print(f'✓ 修复 {fname}')

print('\n✅ 所有补丁已应用完成!')

In [ ]:
#@title 🚀 启动VideoKit
import os
import subprocess
import threading
import time
import re

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['GRADIO_SERVER_PORT'] = '7860'
os.environ['GRADIO_SERVER_NAME'] = '0.0.0.0'

tunnel_url = None

# 启动Cloudflare Tunnel
def start_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line.strip())
        # 提取trycloudflare.com链接
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match and not tunnel_url:
            tunnel_url = match.group()
            print(f"\n{'='*60}")
            print(f"🎉 VideoKit访问链接已生成!")
            print(f"🔗 {tunnel_url}")
            print(f"{'='*60}\n")

tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

# 等待tunnel启动
print("正在启动Cloudflare Tunnel...")
time.sleep(5)

if not tunnel_url:
    print("等待更多时间...")
    time.sleep(5)

# 启动VideoKit应用
print("\n正在启动VideoKit...")
!python videokit.py run

## 💡 使用说明

### 基本操作
1. **SOURCE**: 上传源图像或音频文件
2. **TARGET**: 上传目标视频或图像文件
3. **PROCESSORS**: 选择需要使用的处理器
   - **Style Transfer**: 风格迁移处理
   - **Content Blend**: 内容融合处理
   - **Quality Enhancer**: 画质增强处理
   - **Portrait Editor**: 肖像编辑处理
   - **Frame Colorizer**: 视频上色
   - **Frame Enhancer**: 帧增强
   - **Lip Syncer**: 唇形同步
   - **Expression Restorer**: 表情恢复
   - **Age Modifier**: 年龄修改
   - **Background Remover**: 背景移除
4. **START**: 点击开始处理
5. **OUTPUT**: 下载处理后的结果文件

### 推荐设置
- **视频长度**: 免费Colab建议视频长度不超过10-15秒
- **输出格式**: 音频编码建议选择AAC以获得更好的兼容性
- **分辨率**: 根据GPU显存大小选择合适的分辨率
- **检测角度**: 如果目标视频包含侧面镜头，勾选90度检测角度

## ⚠️ 注意事项

- 免费Colab会话有使用时间限制（通常12小时），长时间运行可能会被中断
- GPU资源由系统动态分配，可能获得T4、P100或V100等不同型号
- 处理大型视频文件时请确保有足够的存储空间（Colab提供约100GB）
- 建议在处理完成后及时下载结果文件到本地
- 首次运行会自动下载模型文件，可能需要几分钟时间
- Cloudflare Tunnel生成的链接每次启动都会变化

## 📝 故障排除

### 常见问题
1. **GPU内存不足**: 尝试降低视频分辨率或缩短视频长度
2. **依赖安装失败**: 尝试重新运行安装代码单元
3. **会话超时**: 保持浏览器窗口打开，或重新连接运行时
4. **连接被拒绝**: 等待Cloudflare Tunnel生成链接（通常需要5-10秒）
5. **模型下载失败**: 检查网络连接，或尝试重新启动运行时
6. **CUDA错误**: 如果遇到CUDA相关错误，可能需要重新安装onnxruntime-gpu

### 重新启动
如果遇到问题，可以点击菜单栏 **Runtime > Restart runtime** 重新启动，然后重新执行所有代码单元。

## 📊 性能参考

| GPU型号 | 显存 | 适用场景 |
|---------|------|----------|
| T4 | 16GB | 标准视频处理 |
| P100 | 16GB | 高清视频处理 |
| V100 | 16/32GB | 大型视频处理 |

*GPU型号由Colab系统自动分配，无法手动选择。*

## 🔗 快速打开

点击以下链接在Colab中打开此笔记本：
```
https://colab.research.google.com/github/tjwcold/video/blob/main/videokit_colab.ipynb
```